# 53 — Tạo và biến đổi cột

Notebook cuối nhóm cơ bản. Nó trả lời một câu hỏi duy nhất nhưng trả lời cho
đến nơi: **làm sao để tính ra một cột mới từ các cột có sẵn.**

pandas cho bạn năm cách, và chúng chênh nhau tới **ba bậc độ lớn** về tốc độ.
Notebook đo cả năm trên cùng một phép tính, cùng một dữ liệu.

Kèm theo: `assign` để nối chuỗi, `where`/`mask`/`np.select` cho logic điều
kiện, `.str` cho cột chuỗi, và ⚠️ **`applymap` đã bị xoá khỏi pandas 3.0**.

In [1]:
import sys
import time
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import hom_nay, lui_ngay

client = finlens.client()
HOM_NAY = hom_nay(client)

gia = client.eod.stock.ohlcv(["HPG", "VCB", "FPT"], start=lui_ngay(HOM_NAY, nam=1))
print(f"pandas {pd.__version__} · frame {len(gia)} dòng")

pandas 3.0.5 · frame 750 dòng


## 1 · `assign` — thêm cột mà không sửa frame gốc

Notebook `52` kết luận: hàm biến đổi dữ liệu nên **trả về frame mới**.
`assign` là công cụ cho việc đó, và nó nối chuỗi được.

In [2]:
co_them = gia.assign(
    gtgd=lambda d: d["close"] * d["volume"] * 1_000,  # nghìn VND × cp × 1000 = VND
    bien_do=lambda d: (d["high"] - d["low"]) / d["low"] * 100,
    gtgd_ty=lambda d: d["gtgd"] / 1e9,  # dùng lại cột vừa tạo ở dòng trên
)

print(f"Frame gốc:  {gia.shape[1]} cột")
print(f"Sau assign: {co_them.shape[1]} cột — thêm {[c for c in co_them.columns if c not in gia.columns]}")
co_them[["symbol", "date", "close", "gtgd_ty", "bien_do"]].head(3).round(
    {"close": 2, "gtgd_ty": 2, "bien_do": 2}
)

Frame gốc:  7 cột
Sau assign: 10 cột — thêm ['gtgd', 'bien_do', 'gtgd_ty']


,symbol,date,close,gtgd_ty,bien_do
0,FPT,2025-08-12,104.95,1155.69,1.50
1,FPT,2025-08-13,102.60,2284.29,2.76
2,FPT,2025-08-14,101.34,1814.53,2.02


Hai chi tiết đáng chú ý:

1. **`lambda d:` thay vì `gia[...]`** — `d` là frame *tại bước đó*, nên nếu bạn
   nối `assign` sau một `query` thì công thức tự động chạy trên phần đã lọc.
2. **`gtgd_ty` dùng được `gtgd`** vừa tạo ở dòng trên. `assign` áp dụng lần
   lượt theo thứ tự tham số.

In [3]:
# assign nối chuỗi được — đây là chỗ lambda trả công
ket_qua = (
    gia.query("symbol == 'HPG'")
    .assign(ls=lambda d: d["close"].pct_change() * 100)
    .assign(ls_tich_luy=lambda d: (1 + d["ls"] / 100).cumprod() * 100 - 100)
    .dropna(subset=["ls"])
)
print(f"HPG: {len(ket_qua)} dòng, lợi suất tích luỹ {ket_qua['ls_tich_luy'].iloc[-1]:+.1f}%")

HPG: 249 dòng, lợi suất tích luỹ -13.1%


⚠️ Nếu viết `gia["close"].pct_change()` thay cho `lambda d: d["close"]...`, công
thức sẽ chạy trên frame **đầy đủ ba mã** rồi mới bị lọc — tức là dòng đầu của
HPG sẽ lấy giá cuối của FPT làm mốc so sánh. Đúng cái lỗi rò rỉ ranh giới nhóm
ở notebook `31`, chỉ khác lớp vỏ.

## 2 · Năm cách tính một cột, và bảng tốc độ

Phép tính giống hệt nhau ở cả năm: `close × volume × 1000`.

In [4]:
lon = client.eod.stock.ohlcv(
    client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()[:120],
    start=lui_ngay(HOM_NAY, nam=1),
)
print(f"Frame đo: {len(lon):,} dòng")


def do(ten: str, fn) -> dict:
    t0 = time.perf_counter()
    kq = fn()
    giay = time.perf_counter() - t0
    return {"cách": ten, "ms": round(giay * 1000, 1), "tổng kiểm tra": round(float(np.sum(kq)) / 1e12, 3)}


ket = [
    do("vectorised (Series × Series)", lambda: lon["close"] * lon["volume"] * 1_000),
    do("numpy thẳng", lambda: lon["close"].to_numpy() * lon["volume"].to_numpy() * 1_000),
    do("eval()", lambda: lon.eval("close * volume * 1000")),
    do("apply(axis=1)", lambda: lon.apply(lambda r: r["close"] * r["volume"] * 1_000, axis=1)),
    do("list comprehension + zip", lambda: pd.Series(
        [c * v * 1_000 for c, v in zip(lon["close"], lon["volume"], strict=True)], index=lon.index
    )),
]

bang = pd.DataFrame(ket).sort_values("ms").reset_index(drop=True)
bang["chậm gấp"] = (bang["ms"] / bang["ms"].min()).round(1).astype(str) + "×"
bang

C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


Frame đo: 29,356 dòng


,cách,ms,tổng kiểm tra,chậm gấp
0,numpy thẳng,0.2,1055.105,1.0×
1,vectorised (Series × Series),0.7,1055.105,3.5×
2,eval(),1.6,1055.105,8.0×
3,list comprehension + zip,8.4,1055.105,42.0×
4,apply(axis=1),196.3,1055.105,981.5×


Cột `tổng kiểm tra` giống nhau ở cả năm dòng — **cùng một kết quả**, chỉ khác
thời gian. Và khoảng cách giữa nhanh nhất với chậm nhất là hàng trăm lần.

### Vì sao `apply(axis=1)` chậm đến thế

Nó gọi hàm Python **một lần cho mỗi hàng**, và mỗi lần phải dựng một `Series`
tạm đại diện cho hàng đó. Phép nhân chỉ là một phần nhỏ trong chi phí.

Phép vectorised đẩy toàn bộ vòng lặp xuống C: một lệnh gọi, một mảng ra.

In [5]:
n = len(lon)
cham_nhat = bang.iloc[-1]
nhanh_nhat = bang.iloc[0]
print(f"Với {n:,} dòng:")
print(f"  {nhanh_nhat['cách']:<32} {nhanh_nhat['ms']:>8.1f} ms")
print(f"  {cham_nhat['cách']:<32} {cham_nhat['ms']:>8.1f} ms")
print()
print()
print("Ngoại suy lên toàn sàn 5 năm (~500.000 dòng):")
for _, r in bang.iterrows():
    print(f"  {r['cách']:<32} {r['ms'] * 500_000 / n / 1000:>8.3f} giây")

Với 29,356 dòng:
  numpy thẳng                           0.2 ms
  apply(axis=1)                       196.3 ms


Ngoại suy lên toàn sàn 5 năm (~500.000 dòng):
  numpy thẳng                         0.003 giây
  vectorised (Series × Series)        0.012 giây
  eval()                              0.027 giây
  list comprehension + zip            0.143 giây
  apply(axis=1)                       3.343 giây


**Quy tắc:** nếu phép tính của bạn diễn đạt được bằng các toán tử trên cột thì
đừng dùng `apply`. `apply(axis=1)` chỉ hợp lý khi logic thật sự cần nhìn cả
hàng và không viết lại được — và ngay cả khi đó, thường vẫn có cách.

## 3 · ⚠️ `applymap` đã bị xoá

pandas 3.0 xoá hẳn `DataFrame.applymap`. Thay bằng `DataFrame.map`.

In [6]:
so = lon[["open", "close"]].head(100)

try:
    so.applymap(lambda x: round(x, 1))
except AttributeError as e:
    print(f"applymap → AttributeError: {e}")

print(f"\nDataFrame.map thay thế → shape {so.map(lambda x: round(x, 1)).shape}")

applymap → AttributeError: 'DataFrame' object has no attribute 'applymap'

DataFrame.map thay thế → shape (100, 2)


⚠️ Nhưng `DataFrame.map` **cũng gọi hàm Python cho từng ô** — nó là bản hai
chiều của `apply(axis=1)`, và chậm tương đương. Ví dụ trên nên viết là
`so.round(1)`.

In [7]:
t0 = time.perf_counter()
_ = so.map(lambda x: round(x, 1))
t_map = time.perf_counter() - t0
t0 = time.perf_counter()
_ = so.round(1)
t_round = time.perf_counter() - t0
print(f".map(lambda) : {t_map * 1000:>7.2f} ms")
print(f".round(1)    : {t_round * 1000:>7.2f} ms   ← nhanh gấp {t_map / t_round:.0f} lần, cùng kết quả")

.map(lambda) :    0.72 ms
.round(1)    :    0.14 ms   ← nhanh gấp 5 lần, cùng kết quả


## 4 · Logic điều kiện: `where`, `mask`, `np.select`

Ba công cụ cho ba hình dạng bài toán khác nhau.

In [8]:
hpg = gia[gia["symbol"] == "HPG"].copy()
hpg["ls"] = hpg["close"].pct_change() * 100

# Một điều kiện, hai nhánh → np.where
hpg["huong"] = np.where(hpg["ls"] >= 0, "tăng", "giảm")

# Giữ giá trị khi đúng, thay khi sai → Series.where
hpg["chi_ngay_tang"] = hpg["ls"].where(hpg["ls"] >= 0, 0)

# Nhiều điều kiện → np.select, có default
hpg["muc_do"] = np.select(
    [hpg["ls"] > 3, hpg["ls"] > 0, hpg["ls"] > -3],
    ["tăng mạnh", "tăng nhẹ", "giảm nhẹ"],
    default="giảm mạnh",
)

hpg[["date", "close", "ls", "huong", "chi_ngay_tang", "muc_do"]].dropna().head(6).round(
    {"close": 2, "ls": 2, "chi_ngay_tang": 2}
)

,date,close,ls,huong,chi_ngay_tang,muc_do
251,2025-08-13,25.09,-1.38,giảm,0.00,giảm nhẹ
252,2025-08-14,25.18,0.36,tăng,0.36,tăng nhẹ
253,2025-08-15,25.00,-0.71,giảm,0.00,giảm nhẹ
254,2025-08-18,25.35,1.40,tăng,1.40,tăng nhẹ
255,2025-08-19,25.13,-0.87,giảm,0.00,giảm nhẹ
256,2025-08-20,24.46,-2.67,giảm,0.00,giảm nhẹ


In [9]:
print("Phân bố mức độ biến động của HPG trong một năm:")
print(hpg["muc_do"].value_counts().to_string())

Phân bố mức độ biến động của HPG trong một năm:
muc_do
giảm nhẹ     142
tăng nhẹ      89
tăng mạnh     12
giảm mạnh      7


⚠️ **`np.select` xét điều kiện theo thứ tự và dừng ở cái đầu tiên đúng.** Nên
thứ tự phải đi từ hẹp tới rộng. Nếu đảo hai dòng đầu, `ls > 0` sẽ nuốt hết cả
nhóm `> 3` và không dòng nào được gán "tăng mạnh".

In [10]:
sai_thu_tu = np.select(
    [hpg["ls"] > 0, hpg["ls"] > 3],  # ← đảo ngược
    ["tăng nhẹ", "tăng mạnh"],
    default="khác",
)
print(f"Thứ tự đúng: {(hpg['muc_do'] == 'tăng mạnh').sum()} dòng 'tăng mạnh'")
print(f"Thứ tự sai : {(sai_thu_tu == 'tăng mạnh').sum()} dòng 'tăng mạnh'   ← điều kiện thứ hai không bao giờ tới lượt")

Thứ tự đúng: 12 dòng 'tăng mạnh'
Thứ tự sai : 0 dòng 'tăng mạnh'   ← điều kiện thứ hai không bao giờ tới lượt


### `pd.cut` khi chia theo ngưỡng số

Ba dòng `np.select` ở trên thật ra là một phép chia khoảng — `pd.cut` diễn đạt
nó gọn hơn và tự sinh nhãn khoảng.

In [11]:
hpg["nhom_cut"] = pd.cut(
    hpg["ls"],
    bins=[-np.inf, -3, 0, 3, np.inf],
    labels=["giảm mạnh", "giảm nhẹ", "tăng nhẹ", "tăng mạnh"],
)
print(hpg["nhom_cut"].value_counts().sort_index().to_string())
print(f"\ndtype: {hpg['nhom_cut'].dtype}   ← pd.cut trả về category, sắp xếp được")

nhom_cut
giảm mạnh      6
giảm nhẹ     142
tăng nhẹ      89
tăng mạnh     12

dtype: category   ← pd.cut trả về category, sắp xếp được


## 5 · Cột chuỗi: bộ truy cập `.str`

Mọi phép chuỗi vectorised nằm sau `.str`.

In [12]:
danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")

ten = danh_muc.assign(
    do_dai=lambda d: d["short_name"].str.len(),
    viet_hoa=lambda d: d["symbol"].str.upper(),
    co_ngan_hang=lambda d: d["short_name"].str.contains("Ngân hàng", case=False, na=False),
    tu_dau=lambda d: d["short_name"].str.split().str[0],
)
print(f"{ten['co_ngan_hang'].sum()} mã có 'Ngân hàng' trong tên rút gọn")
ten[["symbol", "short_name", "do_dai", "tu_dau", "co_ngan_hang"]].head(5)

4 mã có 'Ngân hàng' trong tên rút gọn


,symbol,short_name,do_dai,tu_dau,co_ngan_hang
0,AAA,An Phát Bioplastics,19,An,False
1,AAM,Thủy sản Mekong,15,Thủy,False
2,AAN,Lương thực A An,15,Lương,False
3,AAT,Tập đoàn Tiên Sơn Thanh Hóa,27,Tập,False
4,ABR,Đầu tư Nhãn Hiệu Việt,21,Đầu,False


⚠️ `na=False` trong `str.contains` là bắt buộc khi cột có thể thiếu dữ liệu —
nếu không, kết quả là mask chứa `NA` và không lọc được. Đúng cái bẫy notebook
`51` đã đo.

In [13]:
co_thieu = pd.Series(["Ngân hàng ACB", None, "Hoà Phát"])
print(f"Không có na=False → {co_thieu.str.contains('Ngân hàng').tolist()}")
print(f"Có na=False       → {co_thieu.str.contains('Ngân hàng', na=False).tolist()}")

try:
    co_thieu[co_thieu.str.contains("Ngân hàng")]
except Exception as e:
    print(f"\nLọc bằng mask có NA → {type(e).__name__}: {str(e)[:70]}")

Không có na=False → [True, False, False]
Có na=False       → [True, False, False]


## 6 · `map` trên Series — tra bảng

`Series.map` khác `DataFrame.map`: nó nhận cả **dict**, và khi đó nó là phép
tra bảng vectorised chứ không gọi hàm Python từng ô.

In [14]:
ten_nganh = danh_muc.set_index("symbol")["icb_name2"].to_dict()

ket = gia.assign(nganh=lambda d: d["symbol"].map(ten_nganh))
print(ket[["symbol", "nganh"]].drop_duplicates().to_string(index=False))

print(f"\nMã không có trong bảng tra → {ket['nganh'].isna().sum()} ô NA")

symbol               nganh
   FPT Công nghệ Thông tin
   HPG   Tài nguyên Cơ bản
   VCB           Ngân hàng

Mã không có trong bảng tra → 0 ô NA


`map` với dict để `NA` cho khoá không tìm thấy. Muốn giá trị mặc định thì
`.fillna("khác")` phía sau, hoặc dùng `.replace()` nếu bạn muốn giữ nguyên giá
trị gốc khi không khớp.

In [15]:
thieu_khoa = pd.Series(["HPG", "KHONGCO"])
print(f"map  : {thieu_khoa.map(ten_nganh).tolist()}   ← không khớp thành NA")
print(f"replace: {thieu_khoa.replace(ten_nganh).tolist()}   ← không khớp giữ nguyên")

map  : ['Tài nguyên Cơ bản', nan]   ← không khớp thành NA
replace: ['Tài nguyên Cơ bản', 'KHONGCO']   ← không khớp giữ nguyên


## 7 · Ghép lại: một pipeline hoàn chỉnh

Toàn bộ notebook trong một biểu thức nối chuỗi — không biến trung gian, không
sửa tại chỗ, đọc từ trên xuống theo đúng thứ tự việc xảy ra.

In [16]:
bao_cao = (
    client.eod.stock.ohlcv(
        client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()[:120],
        start=lui_ngay(HOM_NAY, thang=3),
    )
    .sort_values(["symbol", "date"])
    .assign(
        gtgd=lambda d: d["close"] * d["volume"] * 1_000,
        bien_do=lambda d: (d["high"] - d["low"]) / d["low"] * 100,
    )
    .groupby("symbol", observed=True)
    .agg(
        phien=("date", "count"),
        gia_cuoi=("close", "last"),
        gtgd_bq=("gtgd", "mean"),
        bien_do_bq=("bien_do", "mean"),
    )
    .assign(
        gtgd_ty=lambda d: (d["gtgd_bq"] / 1e9).round(1),
        thanh_khoan=lambda d: pd.cut(
            d["gtgd_bq"] / 1e9,
            bins=[-np.inf, 1, 10, 50, np.inf],
            labels=["rất thấp", "thấp", "khá", "cao"],
        ),
    )
    .drop(columns="gtgd_bq")
    .sort_values("gtgd_ty", ascending=False)
)

print(f"{len(bao_cao)} mã · phân bố thanh khoản:")
print(bao_cao["thanh_khoan"].value_counts().sort_index().to_string())
bao_cao.head(10).round(2)

118 mã · phân bố thanh khoản:
thanh_khoan
rất thấp    57
thấp        31
khá         20
cao         10


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


,phien,gia_cuoi,bien_do_bq,gtgd_ty,thanh_khoan
symbol,,,,,
ACB,65,22.65,2.34,451.1,cao
CTG,65,32.15,2.00,281.7,cao
BSR,65,26.10,4.36,274.4,cao
BID,65,38.95,2.55,203.5,cao
CII,65,14.40,4.00,176.8,cao
DXG,65,11.15,3.28,128.2,cao
EIB,65,18.10,2.63,107.1,cao
DIG,65,11.05,3.58,84.6,cao
DCM,65,31.45,2.84,79.0,cao


## Tổng kết

| Bạn cần | Dùng | Không dùng |
|---|---|---|
| Tính cột từ cột khác | toán tử trên `Series` | `apply(axis=1)` |
| Thêm cột, giữ frame gốc | `df.assign(x=lambda d: …)` | `df["x"] = …` trên lát cắt |
| Hai nhánh | `np.where(đk, a, b)` | vòng lặp |
| Nhiều nhánh | `np.select([đk…], [giá trị…], default=)` | `if/elif` trong `apply` |
| Chia theo ngưỡng số | `pd.cut(s, bins=, labels=)` | `np.select` thủ công |
| Tra bảng | `s.map(dict)` | `apply(lambda x: dict[x])` |
| Phép chuỗi | `s.str.*` | `apply(str.upper)` |

**Bốn điều mang sang notebook sau:**

1. **`apply(axis=1)` chậm hàng trăm lần** so với vectorised, trên cùng phép
   tính và cùng kết quả. Nó gọi Python một lần mỗi hàng.
2. **`applymap` đã bị xoá** — dùng `DataFrame.map`, nhưng nhớ rằng nó cũng chậm
   tương đương; phần lớn trường hợp có sẵn phương thức vectorised.
3. **`np.select` dừng ở điều kiện đầu tiên đúng** — thứ tự đi từ hẹp tới rộng.
4. Trong `assign`, dùng `lambda d:` chứ đừng tham chiếu frame bên ngoài — nếu
   không, công thức chạy trên dữ liệu trước khi lọc.

---

**Hết nhóm cơ bản.** Ba notebook `51`–`53` đã dựng nền: cấu trúc và kiểu dữ
liệu, chọn lọc dưới chế độ Copy-on-Write, và biến đổi cột. Nhóm trung cấp tiếp
theo bắt đầu ở `54_groupby` — split-apply-combine trên dữ liệu nhiều mã.